####**Objective**

This notebook prepares the data for machine learning by:

Creating business features\
Creating sentiment drivers\
Encoding labels\
TF-IDF Vectorization\
Train/Test Split\
Class balancing\
Saving artifacts

####Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import joblib
import sys
import importlib # Import importlib

# Add the Shopease directory to the Python path
sys.path.insert(0, 'Shopease')

# Force a reload of the module if it was previously imported
# This is crucial in Jupyter/Colab environments when a source file is modified
module_name = 'src.feature_engineering'
if module_name in sys.modules:
    del sys.modules[module_name]

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import RandomOverSampler

####Driver Dictionary

In [ ]:
# Ensure the directory exists
os.makedirs("Shopease/src", exist_ok=True)

# Create __init__.py to make 'src' a package
init_file_path = "Shopease/src/__init__.py"
with open(init_file_path, "w") as f:
    f.write("")
print("__init__.py created in Shopease/src/")

file_path = "Shopease/src/feature_engineering.py"

content = """
# Feature Engineering Module

def example():
    print("Feature engineering file created successfully!")

driver_keywords = {

    "Delivery":[
        "delivery",
        "driver",
        "parcel",
        "shipping",
        "late",
        "arrived",
        "delay"
    ],

    "Product Quality":[
        "quality",
        "broken",
        "damage",
        "faulty",
        "poor",
        "defective"
    ],

    "Price":[
        "price",
        "cheap",
        "expensive",
        "refund",
        "money"
    ],

    "Customer Support":[
        "support",
        "service",
        "agent",
        "staff",
        "complaint"
    ],

    "Website":[
        "website",
        "app",
        "checkout",
        "payment",
        "login"
    ]

}

def detect_driver(text):

    text = str(text).lower()

    for driver, words in driver_keywords.items():

        for word in words:

            if word in text:

                return driver

    return "Other"
"""

with open(file_path, "w") as f:
    f.write(content)

print("feature_engineering.py created!")

# Import the module after the file is written
import src.feature_engineering

__init__.py created in Shopease/src/
feature_engineering.py created!


####Load Dataset

In [ ]:
df = pd.read_csv(
    "/content/processed_reviews.csv"
)

print(df.shape)

df.head()

(21054, 20)


,review_id,product_category,timestamp,country,rating,review,sentiment,date,year,month,month_name,weekday,hour,review_length,cleaned_review,language,word_count,character_count,sentence_count,avg_word_length
0,REV-50BCBCD9,Sports,2024-09-16 13:44:26+00:00,US,1,"I registered on the website, tried to order a ...",Negative,2024-09-16,2024,9,September,Monday,13,106,registered website tried order laptop entered ...,en,52,364,6,7.000000
1,REV-6D2B2651,Toys,2024-09-16 18:26:46+00:00,GB,1,Had multiple orders one turned up and driver h...,Negative,2024-09-16,2024,9,September,Monday,18,53,multiple order one turned driver phone door nu...,en,31,205,2,6.612903
2,REV-F7E80372,Toys,2024-09-16 21:47:39+00:00,GB,1,I informed these reprobates that I WOULD NOT B...,Negative,2024-09-16,2024,9,September,Monday,21,122,informed reprobate would going visit sick rela...,en,53,351,4,6.622642
3,REV-ED2B173F,Sports,2024-09-17 07:15:49+00:00,AU,1,I have bought from Amazon before and no proble...,Negative,2024-09-17,2024,9,September,Tuesday,7,82,bought amazon problem happy service price amaz...,en,40,278,4,6.950000
4,REV-E48A7AB9,Fashion,2024-09-16 18:37:17+00:00,GB,1,If I could give a lower rate I would! I cancel...,Negative,2024-09-16,2024,9,September,Monday,18,100,could give lower rate would cancelled amazon p...,en,48,337,7,7.020833


####Detect Driver

####Create Sentiment Driver

In [ ]:
# Ensure the module is accessible as an object

df["sentiment_driver"] = (
    df["cleaned_review"]
    .apply(src.feature_engineering.detect_driver)
)

#####check

In [ ]:
df["sentiment_driver"].value_counts()

,count
sentiment_driver,
Delivery,8949
Other,4121
Price,3254
Customer Support,2781
Product Quality,1177
Website,772


####Rating Group

In [ ]:
def rating_group(rating):

    if rating <= 2:

        return "Low"

    elif rating == 3:

        return "Medium"

    else:

        return "High"

In [ ]:
df["rating_group"] = (

    df["rating"]

    .apply(rating_group)

)

####Business Features

In [ ]:
df["review_density"] = (

    df["word_count"]

    /

    df["sentence_count"]

)

In [ ]:
df["weekend"] = (

    df["weekday"]

    .isin(

        ["Saturday","Sunday"]

    )

).astype(int)

In [ ]:
df["long_review"] = (

    df["word_count"] > 50

).astype(int)

####Label Encoding

In [ ]:
label_encoder = LabelEncoder()

df["target"] = label_encoder.fit_transform(

    df["sentiment"]

)

In [ ]:
# Create the 'models' directory if it doesn't exist
os.makedirs("models", exist_ok=True)

joblib.dump(

    label_encoder,

    "models/label_encoder.pkl"

)

['models/label_encoder.pkl']

####Check Mapping

In [ ]:
dict(

zip(

label_encoder.classes_,

label_encoder.transform(

label_encoder.classes_

)

)

)

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}

#####TF-IDF

In [ ]:
tfidf = TfidfVectorizer(

    max_features=10000,

    ngram_range=(1,2),

    min_df=5,

    max_df=0.95,

    stop_words="english"

)

#####Transform

In [ ]:
X = tfidf.fit_transform(

    df["cleaned_review"]

)

#####Target

In [ ]:
y = df["target"]

####Save TF-IDF

In [ ]:
joblib.dump(

    tfidf,

    "models/tfidf_vectorizer.pkl"

)

['models/tfidf_vectorizer.pkl']

####Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)

####Class Balance

#####The dataset is highly imbalanced.
#####Instead of SMOTE, we'll compute class weights.

In [ ]:
weights = compute_class_weight(

    class_weight="balanced",

    classes=np.unique(y_train),

    y=y_train

)

#####Create Dictionary

In [ ]:
class_weights = {

    i:w

    for i,w in enumerate(weights)

}
class_weights

{0: np.float64(0.4890534262485482),
 1: np.float64(7.92984934086629),
 2: np.float64(1.2060866451843895)}

####Save Train Test Data

In [ ]:
joblib.dump(

    X_train,

    "models/X_train.pkl"

)

joblib.dump(

    X_test,

    "models/X_test.pkl"

)

joblib.dump(

    y_train,

    "models/y_train.pkl"

)

joblib.dump(

    y_test,

    "models/y_test.pkl"

)

['models/y_test.pkl']

####Save Final Dataset

In [ ]:
# Create the directory
os.makedirs("data/processed", exist_ok=True)

df.to_csv(

    "data/processed/final_sentiment_dataset.csv",

    index=False

)

#####Save Feature Summary

In [ ]:
feature_summary = pd.DataFrame({

    "Feature":[

        "Review Length",

        "Word Count",

        "Sentence Count",

        "Average Word Length",

        "Review Density",

        "Weekend",

        "Long Review",

        "Sentiment Driver"

    ]

})

In [ ]:
os.makedirs("outputs/reports", exist_ok=True)
feature_summary.to_csv(
    "outputs/reports/feature_summary.csv",
    index=False
)